# 13 — Training and Validation Loops

In the previous notebook, we learned how `Dataset` and `DataLoader` organize data into mini-batches.

Now we will connect everything we have learned so far into one of the most important PyTorch skills:

> **Writing clean training and validation loops**

A complete training system must do more than simply call `loss.backward()`.

It should also:

- Put the model in the correct mode
- Iterate through batches
- Move data to the correct device
- Compute predictions
- Compute loss
- Compute gradients
- Update parameters
- Track metrics correctly
- Evaluate on validation data
- Save the best model
- Avoid accidentally training on validation data

## In this notebook, we will learn:

1. Anatomy of a training loop
2. Training mode
3. Validation mode
4. Forward pass
5. Loss accumulation
6. Backward pass
7. Optimizer step
8. Batch metrics
9. Epoch metrics
10. `model.train()`
11. `model.eval()`
12. `torch.no_grad()`
13. Tracking train vs validation loss
14. Accuracy calculation
15. Best-model checkpointing
16. Early-stopping intuition
17. Clean reusable training functions
18. Common training-loop mistakes
19. Debugging training loops
20. Practice exercises

## Main Goal

By the end of this notebook, you should be able to build the complete loop:

$$
\boxed{
\text{Train Epoch}
\rightarrow
\text{Validation Epoch}
\rightarrow
\text{Track Metrics}
\rightarrow
\text{Save Best Model}
}
$$

The most important principle is:

> **Training updates parameters. Validation measures performance without updating parameters.**


In [ ]:
import copy
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

print("PyTorch version:", torch.__version__)


# 1. The Big Picture

A typical deep-learning workflow contains two separate phases during each epoch:

## Training Phase

$$
\boxed{
train()
\rightarrow
forward
\rightarrow
loss
\rightarrow
backward
\rightarrow
optimizer.step()
}
$$

## Validation Phase

$$
\boxed{
eval()
\rightarrow
no\_grad()
\rightarrow
forward
\rightarrow
loss/metrics
}
$$

The validation phase does **not** update model parameters.


# 2. What Is an Epoch?

An **epoch** means:

> One complete pass through the training dataset.

If the training dataset contains:

$$
1000
$$

samples and:

$$
batch\_size=100
$$

then one epoch contains approximately:

$$
10
$$

training batches.

After all batches have been processed once, the epoch is complete.


# 3. Batch vs Epoch

A **batch** is one small group of samples.

An **epoch** contains many batches.

For example:

$$
\begin{array}{|c|c|}
\hline
\textbf{Quantity} & \textbf{Example} \\
\hline
Dataset\ size & 1000 \\
\hline
Batch\ size & 100 \\
\hline
Batches\ per\ epoch & 10 \\
\hline
\end{array}
$$

The optimizer usually performs one parameter update per training batch.


# 4. Creating a Learnable Synthetic Classification Dataset

We will use a simple binary classification problem with two input features.

The target will depend on:

$$
x_1+x_2
$$

If:

$$
x_1+x_2>0
$$

we assign class:

$$
1
$$

otherwise:

$$
0
$$

This gives us a dataset with a real learnable pattern.


In [ ]:
torch.manual_seed(42)

num_samples = 1000

features = torch.randn(
    num_samples,
    2
)

targets = (
    features[:, 0]
    + features[:, 1]
    > 0
).long()

print("Feature shape:", features.shape)
print("Target shape:", targets.shape)

print(
    "Class 0 samples:",
    (targets == 0).sum().item()
)

print(
    "Class 1 samples:",
    (targets == 1).sum().item()
)


# 5. Train / Validation Split

We will use:

- 80% training
- 20% validation

The split must happen before training.


In [ ]:
split_index = 800

train_features = features[:split_index]
train_targets = targets[:split_index]

val_features = features[split_index:]
val_targets = targets[split_index:]

print(
    "Train:",
    train_features.shape,
    train_targets.shape
)

print(
    "Validation:",
    val_features.shape,
    val_targets.shape
)


# 6. Creating Datasets and DataLoaders

Training data will be shuffled.

Validation data will not be shuffled.


In [ ]:
train_dataset = TensorDataset(
    train_features,
    train_targets
)

val_dataset = TensorDataset(
    val_features,
    val_targets
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

print(
    "Train batches:",
    len(train_loader)
)

print(
    "Validation batches:",
    len(val_loader)
)


# 7. Inspecting One Batch Before Training

Always inspect one batch before building a long loop.


In [ ]:
batch_features, batch_targets = next(
    iter(train_loader)
)

print(
    "Feature batch shape:",
    batch_features.shape
)

print(
    "Target batch shape:",
    batch_targets.shape
)

print(
    "Feature dtype:",
    batch_features.dtype
)

print(
    "Target dtype:",
    batch_targets.dtype
)


For a 2-class problem using `CrossEntropyLoss`:

Model output:

$$
(batch,\ 2)
$$

Target:

$$
(batch)
$$

Target dtype:

`torch.long`


# 8. Building the Model

We will build:

$$
2
\rightarrow
16
\rightarrow
2
$$

The final layer produces two raw logits.


In [ ]:
class BinaryClassifier(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(2, 16),
            nn.ReLU(),
            nn.Linear(16, 2)
        )

    def forward(self, x):
        return self.network(x)

model = BinaryClassifier()

print(model)


# 9. Loss and Optimizer

Because this is standard two-class classification with two logits, we will use:

`CrossEntropyLoss`

Optimizer:

`Adam`


In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01
)

print(criterion)
print(optimizer)


# 10. Device Setup

A reusable training loop should work on either CPU or CUDA.


In [ ]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = model.to(device)

print("Device:", device)


# 11. Anatomy of One Training Batch

For one training batch, the order is:

1. Move batch to device
2. Clear old gradients
3. Forward pass
4. Compute loss
5. Backward pass
6. Optimizer step
7. Compute metrics


In [ ]:
model.train()

batch_features, batch_targets = next(
    iter(train_loader)
)

batch_features = batch_features.to(
    device
)

batch_targets = batch_targets.to(
    device
)

optimizer.zero_grad()

logits = model(
    batch_features
)

loss = criterion(
    logits,
    batch_targets
)

loss.backward()

optimizer.step()

print("Loss:", loss.item())
print("Logit shape:", logits.shape)


# 12. `model.train()`

Before the training phase, call:

```python
model.train()
```

This puts the model into training mode.

It matters for layers such as:

- Dropout
- Batch Normalization

It does **not** mean:

> Start gradient descent automatically.

It only changes module behavior.


In [ ]:
model.train()

print(
    "model.training:",
    model.training
)


# 13. Forward Pass

The forward pass means:

```python
logits = model(inputs)
```

The model transforms the input batch into predictions.

For our problem:

$$
input.shape=(32,\ 2)
$$

and:

$$
logits.shape=(32,\ 2)
$$


In [ ]:
model.train()

batch_features, batch_targets = next(
    iter(train_loader)
)

batch_features = batch_features.to(
    device
)

logits = model(
    batch_features
)

print(
    "Input:",
    batch_features.shape
)

print(
    "Logits:",
    logits.shape
)


# 14. Computing the Loss

For standard multi-class classification:

```python
loss = criterion(
    logits,
    targets
)
```

The loss tensor is usually scalar when the loss uses mean reduction.


In [ ]:
batch_targets = batch_targets.to(
    device
)

loss = criterion(
    logits,
    batch_targets
)

print("Loss:", loss)
print("Loss shape:", loss.shape)


# 15. Backward Pass

The backward pass is:

```python
loss.backward()
```

Autograd computes gradients for trainable parameters.


In [ ]:
optimizer.zero_grad()

logits = model(
    batch_features
)

loss = criterion(
    logits,
    batch_targets
)

loss.backward()

for name, parameter in model.named_parameters():
    print(
        name,
        "| gradient exists:",
        parameter.grad is not None
    )


# 16. Optimizer Step

After gradients have been calculated:

```python
optimizer.step()
```

updates the parameters.

The optimizer uses:

- Current parameter values
- Current gradients
- Optimizer hyperparameters/state


In [ ]:
before = (
    model.network[0]
    .weight
    .detach()
    .clone()
)

optimizer.step()

after = (
    model.network[0]
    .weight
    .detach()
    .clone()
)

print(
    "Weights changed:",
    not torch.equal(
        before,
        after
    )
)


# 17. Why We Clear Gradients

PyTorch gradients accumulate.

Therefore, before the next training batch, use:

```python
optimizer.zero_grad()
```

Without clearing, gradients from multiple batches are added together.

That is only correct if gradient accumulation is intentional.


# 18. Batch Accuracy

For multi-class logits:

$$
(batch,\ classes)
$$

we can obtain predicted class indices using:

```python
predictions = logits.argmax(dim=1)
```

Then compare them with targets.


In [ ]:
with torch.no_grad():
    predictions = logits.argmax(
        dim=1
    )

    correct = (
        predictions
        == batch_targets
    ).sum().item()

    batch_accuracy = (
        correct
        / batch_targets.size(0)
    )

print(
    "Batch accuracy:",
    batch_accuracy
)


# 19. Why Batch Accuracy Alone Is Not Enough

A batch metric describes only one mini-batch.

The next batch may be easier or harder.

We usually want an **epoch metric** based on all samples.

So we accumulate:

- Total loss contribution
- Total correct predictions
- Total sample count

across the entire epoch.


# 20. Loss Accumulation

Suppose the loss function returns mean batch loss.

If:

```python
loss.item()
```

is the average loss for that batch, then to recover the total contribution we multiply by batch size:

$$
batch\ loss\ contribution
=
batch\ mean\ loss
\times
batch\ size
$$

Then across the epoch:

$$
epoch\ loss
=
\frac{
\sum batch\ loss\ contributions
}{
number\ of\ samples
}
$$


In [ ]:
batch_size = batch_targets.size(0)

batch_loss_contribution = (
    loss.item()
    * batch_size
)

print(
    "Batch size:",
    batch_size
)

print(
    "Batch mean loss:",
    loss.item()
)

print(
    "Batch loss contribution:",
    batch_loss_contribution
)


# 21. Why Not Simply Average Batch Losses?

Suppose the last batch is smaller.

If we calculate:

$$
\frac{
loss_1+loss_2+loss_3
}{
3
}
$$

we give the small last batch the same weight as a full batch.

A sample-weighted average is more accurate:

$$
\frac{
loss_1n_1+loss_2n_2+loss_3n_3
}{
n_1+n_2+n_3
}
$$

where:

$$
n_i
$$

is each batch size.


# 22. Building One Training Epoch

Let's write the entire training epoch explicitly.


In [ ]:
model.train()

train_loss_sum = 0.0
train_correct = 0
train_total = 0

for batch_features, batch_targets in train_loader:
    batch_features = batch_features.to(
        device
    )

    batch_targets = batch_targets.to(
        device
    )

    optimizer.zero_grad()

    logits = model(
        batch_features
    )

    loss = criterion(
        logits,
        batch_targets
    )

    loss.backward()

    optimizer.step()

    batch_size = (
        batch_targets.size(0)
    )

    train_loss_sum += (
        loss.item()
        * batch_size
    )

    predictions = logits.argmax(
        dim=1
    )

    train_correct += (
        predictions
        == batch_targets
    ).sum().item()

    train_total += batch_size

train_loss = (
    train_loss_sum
    / train_total
)

train_accuracy = (
    train_correct
    / train_total
)

print(
    "Training loss:",
    train_loss
)

print(
    "Training accuracy:",
    train_accuracy
)


# 23. Training Epoch Metrics

At the end of one training epoch, we now have:

- Average training loss
- Training accuracy

These summarize performance over the entire training dataset.


# 24. Validation Mode

Validation should measure model performance without changing model parameters.

Before validation:

```python
model.eval()
```

This switches layers such as Dropout and BatchNorm into evaluation behavior.


In [ ]:
model.eval()

print(
    "model.training:",
    model.training
)


# 25. Why `model.eval()` Is Necessary

Some layers behave differently during training and validation.

For example:

## Dropout

Training:

- Randomly disables activations

Validation:

- Does not randomly disable activations

## Batch Normalization

Training:

- Uses batch statistics
- Updates running statistics

Validation:

- Uses stored running statistics

Therefore:

> **Always remember the model mode.**


# 26. `torch.no_grad()`

During validation we usually do not need gradients.

Use:

```python
with torch.no_grad():
```

Benefits include:

- Lower memory usage
- Less Autograd overhead
- Clear intent that no backward pass will happen


In [ ]:
model.eval()

batch_features, batch_targets = next(
    iter(val_loader)
)

batch_features = batch_features.to(
    device
)

batch_targets = batch_targets.to(
    device
)

with torch.no_grad():
    logits = model(
        batch_features
    )

    loss = criterion(
        logits,
        batch_targets
    )

print(
    "Requires grad:",
    logits.requires_grad
)


# 27. `eval()` and `no_grad()` Are Different

This distinction is extremely important.

$$
\begin{array}{|c|c|}
\hline
model.eval() & torch.no\_grad() \\
\hline
\text{Changes module behavior} & \text{Disables gradient tracking} \\
\hline
\text{Affects Dropout/BatchNorm} & \text{Reduces Autograd work} \\
\hline
\text{Does not disable Autograd} & \text{Does not change module mode} \\
\hline
\end{array}
$$

For validation, we usually use both.


# 28. Building One Validation Epoch


In [ ]:
model.eval()

val_loss_sum = 0.0
val_correct = 0
val_total = 0

with torch.no_grad():
    for batch_features, batch_targets in val_loader:
        batch_features = batch_features.to(
            device
        )

        batch_targets = batch_targets.to(
            device
        )

        logits = model(
            batch_features
        )

        loss = criterion(
            logits,
            batch_targets
        )

        batch_size = (
            batch_targets.size(0)
        )

        val_loss_sum += (
            loss.item()
            * batch_size
        )

        predictions = logits.argmax(
            dim=1
        )

        val_correct += (
            predictions
            == batch_targets
        ).sum().item()

        val_total += batch_size

val_loss = (
    val_loss_sum
    / val_total
)

val_accuracy = (
    val_correct
    / val_total
)

print(
    "Validation loss:",
    val_loss
)

print(
    "Validation accuracy:",
    val_accuracy
)


# 29. Training vs Validation

The key difference is parameter updating.

$$
\begin{array}{|c|c|}
\hline
\textbf{Training} & \textbf{Validation} \\
\hline
model.train() & model.eval() \\
\hline
Gradients\ enabled & Usually\ no\ gradients \\
\hline
loss.backward() & No\ backward \\
\hline
optimizer.step() & No\ optimizer\ step \\
\hline
Updates\ parameters & Measures\ performance \\
\hline
\end{array}
$$


# 30. Why Validation Data Must Not Update the Model

Validation data is meant to approximate performance on unseen data.

If we update model parameters using validation data, the validation set is no longer independent.

That creates information leakage into model development.

So validation should be used to:

- Monitor
- Compare
- Select checkpoints
- Tune decisions

but not to directly compute optimization updates.


# 31. A Full Multi-Epoch Loop

Now we combine training and validation.


In [ ]:
torch.manual_seed(42)

model = BinaryClassifier().to(
    device
)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01
)

num_epochs = 20

train_losses = []
val_losses = []

train_accuracies = []
val_accuracies = []

for epoch in range(num_epochs):
    # -------------------------
    # Training phase
    # -------------------------
    model.train()

    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    for batch_features, batch_targets in train_loader:
        batch_features = batch_features.to(
            device
        )

        batch_targets = batch_targets.to(
            device
        )

        optimizer.zero_grad()

        logits = model(
            batch_features
        )

        loss = criterion(
            logits,
            batch_targets
        )

        loss.backward()

        optimizer.step()

        batch_size = (
            batch_targets.size(0)
        )

        train_loss_sum += (
            loss.item()
            * batch_size
        )

        predictions = logits.argmax(
            dim=1
        )

        train_correct += (
            predictions
            == batch_targets
        ).sum().item()

        train_total += batch_size

    train_loss = (
        train_loss_sum
        / train_total
    )

    train_accuracy = (
        train_correct
        / train_total
    )

    # -------------------------
    # Validation phase
    # -------------------------
    model.eval()

    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for batch_features, batch_targets in val_loader:
            batch_features = batch_features.to(
                device
            )

            batch_targets = batch_targets.to(
                device
            )

            logits = model(
                batch_features
            )

            loss = criterion(
                logits,
                batch_targets
            )

            batch_size = (
                batch_targets.size(0)
            )

            val_loss_sum += (
                loss.item()
                * batch_size
            )

            predictions = logits.argmax(
                dim=1
            )

            val_correct += (
                predictions
                == batch_targets
            ).sum().item()

            val_total += batch_size

    val_loss = (
        val_loss_sum
        / val_total
    )

    val_accuracy = (
        val_correct
        / val_total
    )

    train_losses.append(
        train_loss
    )

    val_losses.append(
        val_loss
    )

    train_accuracies.append(
        train_accuracy
    )

    val_accuracies.append(
        val_accuracy
    )

    print(
        f"Epoch {epoch + 1:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.3f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.3f}"
    )


# 32. Tracking Train vs Validation Loss

Training loss tells us how well the model fits training data.

Validation loss tells us how well the model performs on unseen validation samples.

Their relationship is often more informative than either one alone.


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(
    train_losses,
    label="Training loss"
)
plt.plot(
    val_losses,
    label="Validation loss"
)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.show()


# 33. Tracking Train vs Validation Accuracy


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    train_accuracies,
    label="Training accuracy"
)
plt.plot(
    val_accuracies,
    label="Validation accuracy"
)
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()
plt.show()


# 34. Interpreting Train and Validation Curves

Some common patterns:

## Both Improve

Good sign.

Training and validation loss both decrease.

## Training Improves but Validation Gets Worse

Possible overfitting.

## Neither Improves

Possible underfitting or optimization problem.

## Training Loss Is Very Noisy

Possible causes:

- Small batches
- Large learning rate
- Noisy data

Curves are diagnostic tools.


# 35. Batch Metrics vs Epoch Metrics

A batch metric may fluctuate a lot.

An epoch metric summarizes the whole dataset.

$$
\begin{array}{|c|c|}
\hline
\textbf{Batch Metric} & \textbf{Epoch Metric} \\
\hline
One\ mini\text{-}batch & Whole\ dataset\ pass \\
\hline
Can\ be\ noisy & More\ stable \\
\hline
Useful\ for\ debugging & Useful\ for\ model\ tracking \\
\hline
\end{array}
$$


# 36. Accuracy Calculation Carefully

For multi-class classification:

```python
predictions = logits.argmax(dim=1)
```

Then:

```python
correct += (
    predictions == targets
).sum().item()
```

At the end:

```python
accuracy = correct / total
```

This gives sample-level accuracy.


# 37. Accuracy Is Not Always the Best Metric

Accuracy can be misleading when classes are imbalanced.

For example, if:

$$
95\%
$$

of samples are class 0, always predicting class 0 gives:

$$
95\%
$$

accuracy but may be useless.

Other metrics can include:

- Precision
- Recall
- Sensitivity
- Specificity
- F1
- AUROC

For now, we use accuracy because it is easy to understand.


# 38. Best-Model Checkpointing

The last epoch is not always the best epoch.

A model may begin to overfit after reaching its best validation performance.

A useful strategy is:

> Save the model whenever validation performance improves.

For example, if monitoring validation loss:

$$
\boxed{
\text{Save when }
val\_loss
<
best\_val\_loss
}
$$


# 39. Saving the Best `state_dict()`

We can keep a copy in memory:

```python
best_state = copy.deepcopy(
    model.state_dict()
)
```

Why `deepcopy`?

Because we want a snapshot of parameter values at that moment.


In [ ]:
best_val_loss = float("inf")
best_state = None

if val_losses[-1] < best_val_loss:
    best_val_loss = val_losses[-1]

    best_state = copy.deepcopy(
        model.state_dict()
    )

print(
    "Saved best validation loss:",
    best_val_loss
)


# 40. Saving a Checkpoint to Disk

A checkpoint can contain more than model weights.

For example:

```python
checkpoint = {
    "epoch": epoch,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "val_loss": val_loss
}
```

Then save with:

```python
torch.save(...)
```


In [ ]:
checkpoint_path = (
    "best_training_checkpoint.pth"
)

checkpoint = {
    "epoch": num_epochs,
    "model_state_dict":
        model.state_dict(),
    "optimizer_state_dict":
        optimizer.state_dict(),
    "val_loss":
        val_losses[-1],
    "val_accuracy":
        val_accuracies[-1]
}

torch.save(
    checkpoint,
    checkpoint_path
)

print(
    "Saved checkpoint:",
    checkpoint_path
)


# 41. Model Weights vs Full Training Checkpoint

These serve different purposes.

## Model Weights Only

```python
torch.save(
    model.state_dict(),
    path
)
```

Useful for:

- Inference
- Deployment
- Loading final weights

## Training Checkpoint

May include:

- Model state
- Optimizer state
- Epoch
- Best metric
- Scheduler state

Useful for:

- Resuming training
- Reproducing training state


# 42. Loading a Training Checkpoint


In [ ]:
loaded_model = BinaryClassifier().to(
    device
)

loaded_optimizer = torch.optim.Adam(
    loaded_model.parameters(),
    lr=0.01
)

loaded_checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False
)

loaded_model.load_state_dict(
    loaded_checkpoint[
        "model_state_dict"
    ]
)

loaded_optimizer.load_state_dict(
    loaded_checkpoint[
        "optimizer_state_dict"
    ]
)

print(
    "Loaded epoch:",
    loaded_checkpoint["epoch"]
)

print(
    "Loaded validation loss:",
    loaded_checkpoint["val_loss"]
)


# 43. Best Checkpoint vs Resume Checkpoint

Sometimes we save two different types of checkpoints:

## Best Model

Used for final evaluation or deployment.

Saved when validation metric improves.

## Latest Training State

Used to resume interrupted training.

Saved periodically or every epoch.

These are related but not always the same checkpoint.


# 44. Early Stopping Intuition

Early stopping stops training when validation performance has not improved for some number of epochs.

The number of allowed non-improving epochs is often called:

> **Patience**

Example:

$$
patience=5
$$

means:

> Stop if validation loss does not improve for 5 consecutive epochs.


# 45. Why Early Stopping Can Help

Training longer is not always better.

A model may eventually begin overfitting.

Early stopping can:

- Save computation
- Reduce unnecessary training
- Stop near the best validation region

However, it is still a model-selection decision and should be based on validation data, not test data.


# 46. Basic Early-Stopping Logic

Conceptually:

```python
if val_loss < best_val_loss:
    best_val_loss = val_loss
    epochs_without_improvement = 0
else:
    epochs_without_improvement += 1

if epochs_without_improvement >= patience:
    stop_training = True
```


In [ ]:
best_val_loss = float("inf")
epochs_without_improvement = 0
patience = 3

example_val_losses = [
    0.8,
    0.6,
    0.5,
    0.51,
    0.52,
    0.53
]

for epoch, current_val_loss in enumerate(
    example_val_losses,
    start=1
):
    if current_val_loss < best_val_loss:
        best_val_loss = (
            current_val_loss
        )

        epochs_without_improvement = 0

        print(
            f"Epoch {epoch}: improved"
        )

    else:
        epochs_without_improvement += 1

        print(
            f"Epoch {epoch}: "
            f"no improvement "
            f"({epochs_without_improvement})"
        )

    if (
        epochs_without_improvement
        >= patience
    ):
        print(
            "Early stopping triggered."
        )
        break


# 47. Minimum Improvement — `min_delta`

Sometimes tiny numerical changes should not count as meaningful improvement.

We can define:

$$
min\_delta
$$

For example:

```python
current_loss <
best_loss - min_delta
```

This prevents extremely small fluctuations from constantly resetting patience.


In [ ]:
best_loss = 1.0
current_loss = 0.99995
min_delta = 0.001

meaningful_improvement = (
    current_loss
    < best_loss - min_delta
)

print(
    "Meaningful improvement:",
    meaningful_improvement
)


# 48. Reusable Training Function

Now let's write a clean function for one training epoch.


In [ ]:
def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device
):
    model.train()

    loss_sum = 0.0
    correct = 0
    total = 0

    for inputs, targets in loader:
        inputs = inputs.to(
            device
        )

        targets = targets.to(
            device
        )

        optimizer.zero_grad()

        logits = model(
            inputs
        )

        loss = criterion(
            logits,
            targets
        )

        loss.backward()

        optimizer.step()

        batch_size = (
            targets.size(0)
        )

        loss_sum += (
            loss.item()
            * batch_size
        )

        predictions = logits.argmax(
            dim=1
        )

        correct += (
            predictions
            == targets
        ).sum().item()

        total += batch_size

    epoch_loss = (
        loss_sum
        / total
    )

    epoch_accuracy = (
        correct
        / total
    )

    return (
        epoch_loss,
        epoch_accuracy
    )


# 49. Reusable Validation Function


In [ ]:
def validate_one_epoch(
    model,
    loader,
    criterion,
    device
):
    model.eval()

    loss_sum = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, targets in loader:
            inputs = inputs.to(
                device
            )

            targets = targets.to(
                device
            )

            logits = model(
                inputs
            )

            loss = criterion(
                logits,
                targets
            )

            batch_size = (
                targets.size(0)
            )

            loss_sum += (
                loss.item()
                * batch_size
            )

            predictions = logits.argmax(
                dim=1
            )

            correct += (
                predictions
                == targets
            ).sum().item()

            total += batch_size

    epoch_loss = (
        loss_sum
        / total
    )

    epoch_accuracy = (
        correct
        / total
    )

    return (
        epoch_loss,
        epoch_accuracy
    )


# 50. Testing the Reusable Functions


In [ ]:
torch.manual_seed(42)

reusable_model = BinaryClassifier().to(
    device
)

reusable_criterion = (
    nn.CrossEntropyLoss()
)

reusable_optimizer = torch.optim.Adam(
    reusable_model.parameters(),
    lr=0.01
)

train_loss, train_acc = (
    train_one_epoch(
        reusable_model,
        train_loader,
        reusable_criterion,
        reusable_optimizer,
        device
    )
)

val_loss, val_acc = (
    validate_one_epoch(
        reusable_model,
        val_loader,
        reusable_criterion,
        device
    )
)

print(
    "Train:",
    train_loss,
    train_acc
)

print(
    "Validation:",
    val_loss,
    val_acc
)


# 51. Clean `fit()` Function With Best Checkpointing

Now we can build a higher-level reusable training function.

This version will:

- Train for multiple epochs
- Validate every epoch
- Track history
- Keep the best model state
- Support early stopping


In [ ]:
def fit(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    epochs=20,
    patience=None,
    min_delta=0.0
):
    history = {
        "train_loss": [],
        "train_accuracy": [],
        "val_loss": [],
        "val_accuracy": []
    }

    best_val_loss = float("inf")

    best_state = copy.deepcopy(
        model.state_dict()
    )

    epochs_without_improvement = 0

    for epoch in range(epochs):
        train_loss, train_acc = (
            train_one_epoch(
                model,
                train_loader,
                criterion,
                optimizer,
                device
            )
        )

        val_loss, val_acc = (
            validate_one_epoch(
                model,
                val_loader,
                criterion,
                device
            )
        )

        history[
            "train_loss"
        ].append(train_loss)

        history[
            "train_accuracy"
        ].append(train_acc)

        history[
            "val_loss"
        ].append(val_loss)

        history[
            "val_accuracy"
        ].append(val_acc)

        print(
            f"Epoch {epoch + 1:02d} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_acc:.3f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.3f}"
        )

        improved = (
            val_loss
            < best_val_loss - min_delta
        )

        if improved:
            best_val_loss = (
                val_loss
            )

            best_state = copy.deepcopy(
                model.state_dict()
            )

            epochs_without_improvement = 0

        else:
            epochs_without_improvement += 1

        if (
            patience is not None
            and
            epochs_without_improvement
            >= patience
        ):
            print(
                "Early stopping."
            )
            break

    model.load_state_dict(
        best_state
    )

    return history


# 52. Running the Clean Training Function


In [ ]:
torch.manual_seed(42)

final_model = BinaryClassifier().to(
    device
)

final_criterion = (
    nn.CrossEntropyLoss()
)

final_optimizer = torch.optim.Adam(
    final_model.parameters(),
    lr=0.01
)

history = fit(
    model=final_model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=final_criterion,
    optimizer=final_optimizer,
    device=device,
    epochs=30,
    patience=5,
    min_delta=1e-4
)


# 53. Why Reload the Best State?

At the end of training, the model currently in memory corresponds to the last epoch that ran.

But the best validation performance may have happened earlier.

So after training:

```python
model.load_state_dict(
    best_state
)
```

restores the best validation checkpoint.


# 54. Plotting History From the Reusable Function


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    history["train_loss"],
    label="Training loss"
)
plt.plot(
    history["val_loss"],
    label="Validation loss"
)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss History")
plt.legend()
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    history["train_accuracy"],
    label="Training accuracy"
)
plt.plot(
    history["val_accuracy"],
    label="Validation accuracy"
)
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy History")
plt.legend()
plt.show()


# 55. Saving the Best Final Model

After `fit()` reloads the best state, we can save:

```python
model.state_dict()
```


In [ ]:
best_model_path = (
    "best_binary_classifier.pth"
)

torch.save(
    final_model.state_dict(),
    best_model_path
)

print(
    "Saved:",
    best_model_path
)


# 56. Evaluating the Best Model

We can run validation again after the best state has been restored.


In [ ]:
best_val_loss, best_val_accuracy = (
    validate_one_epoch(
        final_model,
        val_loader,
        final_criterion,
        device
    )
)

print(
    "Best validation loss:",
    best_val_loss
)

print(
    "Best validation accuracy:",
    best_val_accuracy
)


# 57. Binary Classification With One Logit

Not every binary classifier uses two logits.

Another valid design uses:

$$
1
$$

output logit per sample and:

`BCEWithLogitsLoss`

Then:

$$
logits.shape=(batch,\ 1)
$$

and targets are commonly:

$$
(batch,\ 1)
$$

floating-point values.

Metric code must match the output format.


# 58. Accuracy for One-Logit Binary Classification

For one-logit binary classification:

1. Apply sigmoid
2. Apply threshold
3. Compare with target


In [ ]:
example_logits = torch.tensor([
    [-2.0],
    [0.2],
    [1.5],
    [0.8]
])

example_targets = torch.tensor([
    [0.0],
    [1.0],
    [1.0],
    [1.0]
])

probabilities = torch.sigmoid(
    example_logits
)

predictions = (
    probabilities >= 0.5
).float()

accuracy = (
    predictions
    == example_targets
).float().mean()

print(
    "Accuracy:",
    accuracy.item()
)


# 59. Metric Logic Must Match the Task

Different tasks require different metric logic.

## Multi-Class

```python
predictions = logits.argmax(dim=1)
```

## Binary, One Logit

```python
probabilities = sigmoid(logits)
predictions = probabilities >= threshold
```

## Regression

Accuracy is usually not meaningful.

Possible metrics include:

- MAE
- RMSE
- $R^2$

The training loop should be designed around the task.


# 60. Loss and Metric Are Different

The loss is used for optimization.

The metric is used for interpretation and evaluation.

For example:

$$
\begin{array}{|c|c|}
\hline
\textbf{Task} & \textbf{Possible Loss / Metric} \\
\hline
Regression & MSE\ loss,\ MAE\ metric \\
\hline
Binary & BCE\ loss,\ Accuracy/F1 \\
\hline
Multi\text{-}class & CrossEntropy,\ Accuracy \\
\hline
\end{array}
$$

Do not assume the training loss must equal the evaluation metric.


# 61. Common Mistake — Forgetting `model.train()`

If the previous phase called:

`model.eval()`

and the next epoch begins without:

`model.train()`

layers such as Dropout or BatchNorm may remain in evaluation behavior during training.

Always explicitly switch modes.


# 62. Common Mistake — Forgetting `model.eval()`

Validation performed while the model remains in training mode can give inconsistent behavior when using:

- Dropout
- BatchNorm

Always call:

```python
model.eval()
```

before validation.


# 63. Common Mistake — Forgetting `torch.no_grad()`

Validation can still run without `torch.no_grad()`, but PyTorch will build computational graphs unnecessarily.

This wastes memory and computation.

A clean validation loop uses:

```python
with torch.no_grad():
```


# 64. Common Mistake — Calling `optimizer.step()` During Validation

Never do:

```python
optimizer.step()
```

in a normal validation loop.

Validation should not change model parameters.


# 65. Common Mistake — Incorrect Loss Averaging

This is often inaccurate:

```python
epoch_loss = sum(batch_losses) / number_of_batches
```

when batches have different sizes.

A safer sample-weighted calculation is:

```python
loss_sum += (
    loss.item()
    * batch_size
)

epoch_loss = (
    loss_sum / total_samples
)
```


# 66. Common Mistake — Dividing Correct Predictions by Number of Batches

Accuracy should be:

$$
\frac{
correct\ predictions
}{
total\ samples
}
$$

not:

$$
\frac{
correct\ predictions
}{
number\ of\ batches
}
$$


# 67. Common Mistake — Saving Only the Last Model

The final model may not be the best model.

Track validation performance and save the best checkpoint according to the metric that matters for the task.

Possible checkpoint criteria include:

- Lowest validation loss
- Highest validation accuracy
- Highest validation F1
- Highest validation AUROC

Choose the criterion before looking at test performance.


# 68. Common Mistake — Using Test Data for Early Stopping

Early stopping should use validation data.

The test set should remain untouched until final evaluation.

Otherwise the test set becomes part of model selection.


# 69. Common Mistake — Using the Wrong Metric Direction

Some metrics are better when smaller:

- Loss
- MAE
- RMSE

Some are better when larger:

- Accuracy
- F1
- AUROC

Checkpoint logic must know whether to minimize or maximize the chosen metric.


# 70. Common Mistake — Forgetting Device Movement

If the model is on GPU but the batch is on CPU, the forward pass will fail.

A clean loop moves:

```python
inputs = inputs.to(device)
targets = targets.to(device)
```

before the forward pass.


# 71. Common Mistake — Storing Tensor Losses Instead of Python Numbers

If you append the full loss tensor every epoch:

```python
losses.append(loss)
```

you may accidentally keep graph references in some situations.

For logging, usually store:

```python
loss.item()
```

after the needed backward computation.


# 72. Training-Loop Debugging Checklist

If training does not work, inspect:

1. Is the model in training mode?
2. Is validation in evaluation mode?
3. Are inputs and targets on the correct device?
4. Are prediction and target shapes correct?
5. Is the correct loss being used?
6. Is `optimizer.zero_grad()` called?
7. Is `loss.backward()` called?
8. Is `optimizer.step()` called?
9. Are gradients finite?
10. Are parameters changing?
11. Is training loss decreasing?
12. Is validation loss reasonable?
13. Is metric calculation correct?
14. Is validation accidentally updating parameters?
15. Is the best checkpoint being tracked correctly?


In [ ]:
for name, parameter in final_model.named_parameters():
    print(
        name,
        "| grad exists:",
        parameter.grad is not None
    )


# 73. Checking Gradient Magnitudes

Extremely small, huge, or non-finite gradients can indicate training problems.


In [ ]:
for name, parameter in final_model.named_parameters():
    if parameter.grad is not None:
        grad_norm = (
            parameter.grad
            .norm()
            .item()
        )

        print(
            name,
            "| grad norm:",
            grad_norm
        )


# 74. Checking Whether Parameters Change

A useful debugging strategy:

1. Copy one parameter
2. Run a training step
3. Compare before and after


In [ ]:
debug_model = BinaryClassifier().to(
    device
)

debug_optimizer = torch.optim.Adam(
    debug_model.parameters(),
    lr=0.01
)

debug_criterion = (
    nn.CrossEntropyLoss()
)

inputs, targets = next(
    iter(train_loader)
)

inputs = inputs.to(
    device
)

targets = targets.to(
    device
)

before = (
    debug_model.network[0]
    .weight
    .detach()
    .clone()
)

debug_model.train()

debug_optimizer.zero_grad()

logits = debug_model(
    inputs
)

loss = debug_criterion(
    logits,
    targets
)

loss.backward()

debug_optimizer.step()

after = (
    debug_model.network[0]
    .weight
    .detach()
    .clone()
)

print(
    "Parameter changed:",
    not torch.equal(
        before,
        after
    )
)


# 75. Practice Exercises

Try solving these before looking at the solutions.

## Exercise 1

Write the five core operations of a training batch in the correct order.

## Exercise 2

Explain the difference between:

`model.train()`

and:

`model.eval()`

## Exercise 3

Why is:

`torch.no_grad()`

normally used during validation?

## Exercise 4

Write a training epoch that returns:

- Average loss
- Accuracy

## Exercise 5

Write a validation epoch that returns:

- Average loss
- Accuracy

## Exercise 6

Track training and validation loss for 10 epochs.

## Exercise 7

Save the best model based on minimum validation loss.

## Exercise 8

Implement early stopping with:

$$
patience=3
$$

## Exercise 9

For a binary one-logit classifier, write the accuracy calculation.

## Exercise 10

Explain why test data should not be used for checkpoint selection.


# 76. Conceptual Challenges

Answer before running code.

## Challenge 1

What happens if `optimizer.step()` is called during validation?

## Challenge 2

What happens if `model.eval()` is forgotten when using Dropout?

## Challenge 3

Why can averaging batch losses equally be wrong?

## Challenge 4

Why should validation loss sometimes increase while training loss continues decreasing?

## Challenge 5

Why is the lowest validation loss often a useful checkpoint criterion?

## Challenge 6

What is the difference between best-model checkpointing and early stopping?

## Challenge 7

Why can accuracy be misleading on imbalanced data?

## Challenge 8

Why should the model be returned to `train()` mode after validation before the next training epoch?


# 77. Exercise Solutions


In [ ]:
# Exercise 1
print(
    "Exercise 1:"
)

print(
    "zero_grad -> forward -> loss -> backward -> step"
)

# Exercise 4
def exercise_train_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device
):
    model.train()

    loss_sum = 0.0
    correct = 0
    total = 0

    for x_batch, y_batch in loader:
        x_batch = x_batch.to(
            device
        )

        y_batch = y_batch.to(
            device
        )

        optimizer.zero_grad()

        logits = model(
            x_batch
        )

        loss = criterion(
            logits,
            y_batch
        )

        loss.backward()

        optimizer.step()

        batch_size = (
            y_batch.size(0)
        )

        loss_sum += (
            loss.item()
            * batch_size
        )

        correct += (
            logits.argmax(dim=1)
            == y_batch
        ).sum().item()

        total += batch_size

    return (
        loss_sum / total,
        correct / total
    )

# Exercise 5
def exercise_val_epoch(
    model,
    loader,
    criterion,
    device
):
    model.eval()

    loss_sum = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for x_batch, y_batch in loader:
            x_batch = x_batch.to(
                device
            )

            y_batch = y_batch.to(
                device
            )

            logits = model(
                x_batch
            )

            loss = criterion(
                logits,
                y_batch
            )

            batch_size = (
                y_batch.size(0)
            )

            loss_sum += (
                loss.item()
                * batch_size
            )

            correct += (
                logits.argmax(dim=1)
                == y_batch
            ).sum().item()

            total += batch_size

    return (
        loss_sum / total,
        correct / total
    )

# Exercise 9
binary_logits = torch.tensor([
    [-1.0],
    [2.0],
    [0.3]
])

binary_targets = torch.tensor([
    [0.0],
    [1.0],
    [1.0]
])

binary_predictions = (
    torch.sigmoid(
        binary_logits
    )
    >= 0.5
).float()

binary_accuracy = (
    binary_predictions
    == binary_targets
).float().mean()

print(
    "Exercise 9 accuracy:",
    binary_accuracy.item()
)


# 78. Key Takeaways

In this notebook, we learned:

- Anatomy of a training loop
- Epochs and batches
- Training mode
- Validation mode
- Forward pass
- Loss calculation
- Loss accumulation
- Backward pass
- Optimizer step
- Batch accuracy
- Epoch accuracy
- `model.train()`
- `model.eval()`
- `torch.no_grad()`
- Training vs validation loss
- Training vs validation accuracy
- Best-model checkpointing
- Full training checkpoints
- Early-stopping intuition
- Patience
- `min_delta`
- Reusable training functions
- Reusable validation functions
- A clean `fit()` function
- Common loop mistakes
- Training-loop debugging

The central pattern is:

$$
\boxed{
\begin{array}{c}
\textbf{Training:}\\
train()
\rightarrow
zero\_grad()
\rightarrow
forward
\rightarrow
loss
\rightarrow
backward()
\rightarrow
step()
\end{array}
}
$$

and:

$$
\boxed{
\begin{array}{c}
\textbf{Validation:}\\
eval()
\rightarrow
no\_grad()
\rightarrow
forward
\rightarrow
loss/metrics
\end{array}
}
$$


# 79. Check Your Understanding

Before moving forward, make sure you can answer these without searching:

1. What is an epoch?
2. What is a training batch?
3. What does `model.train()` do?
4. What does `model.eval()` do?
5. Why is `torch.no_grad()` used during validation?
6. What is the forward pass?
7. What does `loss.backward()` do?
8. What does `optimizer.step()` do?
9. Why must gradients be cleared?
10. How should average epoch loss be calculated when batch sizes differ?
11. How is multi-class accuracy calculated?
12. Why is validation different from training?
13. Why should validation not update parameters?
14. What does training-vs-validation loss tell us?
15. What is best-model checkpointing?
16. Why might the last epoch not be the best epoch?
17. What is early stopping?
18. What does patience mean?
19. What is `min_delta`?
20. Why should the test set not be used for early stopping?


# Next Notebook

# 14 — Multi-Layer Perceptron for Classification

In the next notebook, we will study:

- What is an MLP?
- Input, hidden, and output layers
- Choosing hidden dimensions
- ReLU activations
- Logits
- Multi-class classification
- Synthetic classification data
- Complete MLP with `nn.Module`
- Training with DataLoader
- Validation loop
- Accuracy
- Confusion-matrix intuition
- Decision boundaries
- Overfitting intuition
- Improving an MLP
